# Notebook 5 — CNN on Rice Images (R4)
Rice Variety Classification | F20DL 2026/2027

**Dataset:** Kaggle Rice Image Dataset — 5 classes, 75,000 images
Download: `kaggle datasets download -d muratkokludataset/rice-image-dataset`
Extract to: `../data/images/`

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = '../data/images/Rice_Image_Dataset'
IMG_SIZE = 64
BATCH_SIZE = 64

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_ds, test_ds = torch.utils.data.random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=2)
print('Classes:', dataset.classes)

## 1. Define CNN

In [ ]:
class RiceCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RiceCNN(num_classes=len(dataset.classes)).to(device)
print(model)

## 2. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

train_losses, val_accs = [], []
for epoch in range(15):
    model.train()
    epoch_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    train_losses.append(epoch_loss / len(train_dl))

    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
    val_accs.append(correct / len(test_ds))
    print(f'Epoch {epoch+1}: loss={train_losses[-1]:.4f}, acc={val_accs[-1]:.4f}')

## 3. Plot & Save

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses); ax1.set_title('Training Loss')
ax2.plot(val_accs); ax2.set_title('Validation Accuracy')
plt.tight_layout(); plt.savefig('../results/cnn_training.png', dpi=100); plt.show()
torch.save(model.state_dict(), '../models/cnn.pt')